<a href="https://colab.research.google.com/github/JHastings46/Data-Science-Interview-Assistant/blob/main/1_DataAssistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Generative AI Use Case: Summarize Dialogue**

For this project I will build a **Data Science Interview assitant **. I will first do the Data Science interview question task using generative AI. I will explore how the input text affects the output of the model, and perform prompt engineering to direct it towards the task I need. By comparing zero shot, one shot, and few shot inferences, I will take the first step towards prompt engineering and see how it can enhance the generative output of Large Language Models.



I will install the required packages to use PyTorch and Hugging Face transformers and datasets.

In [ ]:
# First upgrade pip
%pip install --upgrade pip

# Install torch and torchdata
%pip install --no-deps torch==2.5.1 torchdata==0.6.0 --quiet

# Then install other packages except TRL
%pip install -U \
    datasets==2.17.0 \
    transformers==4.38.2 \
    evaluate==0.4.0 \
    rouge_score==0.1.2 \
    peft==0.3.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 88.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.10.0 which is incompatible.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.5.1 which is incompatible.
torchtune 0.6.1 requires torchdata==0.11.0, but you have torchdata 0.6.0 which is incompatible.


Load the datasets, Large Language Model (LLM), tokenizer, and configurator. Do not worry if you do not understand yet all of those components - they will be described and discussed later in the notebook.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

###**Summarize Questions without Prompt Engineering**

In this use case, I will be generating a summary of interview questions with the pre-trained Large Language Model (LLM) FLAN-T5 from Hugging Face. The list of available models in the Hugging Face transformers package can be found [here](https://huggingface.co/docs/transformers/index).

Let's upload some simple dialogues from the https://huggingface.co/datasets/UdayG01/DataScienceInterviewQuestions. This dataset contains < 1K questions with answers.

In [ ]:
huggingface_dataset_name = "UdayG01/DataScienceInterviewQuestions"

dataset = load_dataset(huggingface_dataset_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/47 [00:00<?, ? examples/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Question', 'Answer'],
        num_rows: 47
    })
})


In [ ]:
print(dataset.keys())

dict_keys(['train'])


Print a couple of questions with their baseline answers.

In [ ]:
example_indices = [0, 1]

dash_line = '-'.join('' for x in range(100))

for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print('INPUT QUESTION:')
    print(dataset['train'][index]['Question'])
    print(dash_line)
    print('BASELINE HUMAN ANSWER:')
    print(dataset['train'][index]['Answer'])
    print(dash_line)
    print()

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT QUESTION:
### Question:
Discuss the concept of dimensionality reduction and mention a popular technique for achieving it.

### Answer:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Dimensionality reduction aims to reduce the number of features in a dataset while preserving its important information. Principal Component Analysis (PCA) is a popular technique for dimensionality reduction...
---------------------------------------------------------------------------------------------------

---------------------------------------------------------------------------------------------------
Example  2
---------------------------------------------------------------------------------------------------
IN

This cell loads the [FLAN-T5 model](https://huggingface.co/docs/transformers/model_doc/flan-t5) from Hugging Face by creating an instance of the `AutoModelForSeq2SeqLM`class with the. `.from_pretrained()` method. FLAN-T5 is a text-to-text model, which means it takes text as input and creates text as output. This makes it useful for summarizing conversations, answering questions, and explaining concepts.

For my data science interview assistant, this model acts like the “brain” that reads my prompt and generates the answer.


In [ ]:
model_name='google/flan-t5-base'

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

This loads the FLAN-T5 tokenizer by creating an instance with **AutoTokenizer.from_pretrained()**. The tokenizer performs **encoding**, which turns normal text into token numbers the model can understand, and **decoding**, which turns the model’s token numbers back into readable text. The **use_fast=True** setting uses the faster tokenizer version when available. For my data science interview assistant, this helps convert interview questions into model input and convert the model’s generated answer back into plain English.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [ ]:
#Test the tokenizer encoding and decoding a simple sentence:
sentence = "Whats the best Python library to use?"

sentence_encoded = tokenizer(sentence, return_tensors='pt')

sentence_decoded = tokenizer.decode(
        sentence_encoded["input_ids"][0],
        skip_special_tokens=True
    )

print('ENCODED SENTENCE:')
print(sentence_encoded["input_ids"][0])
print('\nDECODED SENTENCE:')
print(sentence_decoded)

ENCODED SENTENCE:
tensor([  363,     7,     8,   200, 20737,  3595,    12,   169,    58,     1])

DECODED SENTENCE:
Whats the best Python library to use?


This step tests how well the base LLM answers or summarizes a question without changing the prompt to help it. This means we first give the model the question by itself and see what kind of answer it creates. **Prompt engineering** means changing the input, such as adding clearer instructions or examples, to improve the model’s response. For my data science interview assistant, this gives me a baseline answer first so I can compare it later against better zero-shot, one-shot, or few-shot prompts.


In [ ]:
for i, index in enumerate(example_indices):
    question = dataset['train'][index]['Question']
    answer = dataset['train'][index]['Answer']

    inputs = tokenizer(question, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{question}')
    print(dash_line)
    print(f'BASELINE HUMAN ANSWER:\n{answer}')
    print(dash_line)
    print(f'MODEL GENERATION - WITHOUT PROMPT ENGINEERING:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
### Question:
Discuss the concept of dimensionality reduction and mention a popular technique for achieving it.

### Answer:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Dimensionality reduction aims to reduce the number of features in a dataset while preserving its important information. Principal Component Analysis (PCA) is a popular technique for dimensionality reduction...
---------------------------------------------------------------------------------------------------
MODEL GENERATION - WITHOUT PROMPT ENGINEERING:
a dimensional refraction

---------------------------------------------------------------------------------------------------
Example  2
--------------------------------

The model’s answers make partial sense, but the first answer shows the model is not fully clear on the task. For the dimensionality reduction question, it gave a strange phrase instead of a real explanation. For the elbow method question, it gave a better answer, but it was still too short compared with the human answer. This shows why prompt engineering matters: clearer instructions can tell the model to explain the concept fully, use simple language, and include the key technique or example.

###**Zero-Shot Inference with an Instruction Prompt**

This section is about improving the model’s answer by turning the question into a clear instruction prompt. Zero-shot inference means the model gets the question plus instructions, but no example answer first. Instead of only giving the model a question, you wrap it with directions like **“Answer the following data science interview question in simple terms.”** For my interview assistant, this helps the model understand that its job is to explain ML/statistics concepts clearly, not just guess a short next sentence.

In [ ]:
for i, index in enumerate(example_indices):
    raw_question = dataset['train'][index]['Question']
    answer = dataset['train'][index]['Answer']

    question = raw_question.replace("### Question:", "").replace("### Answer:", "").strip()

    prompt = f"""
Explain this machine learning concept like I am a beginner.

Question:
{question}

Answer:
"""
    # Input constructed prompt instead of the question.
    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN ANSWER:\n{answer}')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Explain this machine learning concept like I am a beginner.

Question:
Discuss the concept of dimensionality reduction and mention a popular technique for achieving it.

Answer:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Dimensionality reduction aims to reduce the number of features in a dataset while preserving its important information. Principal Component Analysis (PCA) is a popular technique for dimensionality reduction...
---------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
a dimensionality reduction algorithm

----------------------------------------------------------------------------------------------

Changing the prompt to **“Explain this machine learning concept like I am a beginner”** slightly improved the model’s focus, but the answers were still too short and incomplete, showing that the model needs stronger instructions or examples.

###**Zero Shot Inference with the Prompt Template from FLAN-T5**

Let's use a slightly different prompt. FLAN-T5 has many prompt templates that are published for certain tasks [here](https://github.com/google-research/FLAN/tree/main/flan/v2). In the following code, I will use one of the [pre-built FLAN-T5 prompts](https://github.com/google-research/FLAN/blob/main/flan/v2/templates.py):

In [ ]:
for i, index in enumerate(example_indices):
    raw_question = dataset['train'][index]['Question']
    answer = dataset['train'][index]['Answer']

    question = raw_question.replace("### Question:", "").replace("### Answer:", "").strip()

    prompt = f"""
Define all data science terms and concepts.
Question:
{question}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"], max_new_tokens=50)[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN ANSWER:\n{answer}\n')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Define all data science terms and concepts.
Question:
Discuss the concept of dimensionality reduction and mention a popular technique for achieving it.

Answer:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Dimensionality reduction aims to reduce the number of features in a dataset while preserving its important information. Principal Component Analysis (PCA) is a popular technique for dimensionality reduction...

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
dimensionality reduction is the process of reducing the size of an object by reducing the size of the object.

-------------------------------------

###**Summarize Questions with One Shot and Few Shot Inference**

This section teaches the model by showing it solved examples before asking a new question. One-shot means the model sees 1 example question with a strong answer, then answers a new question. Few-shot means the model sees 2 or more example question-answer pairs before the new question. This is called in-context learning because the model learns the answer style from the examples inside the prompt without being retrained.

We build a helper function that creates a one-shot prompt automatically. The function first adds one full solved example: a question plus its human answer. Then it adds a new question at the end without the answer, so the model has to complete it. For my data science interview assistant, this helps me show the model one strong interview answer first, then ask it to answer a new ML/statistics question in the same style.

In [ ]:
def make_prompt(example_indices_full, example_index_to_answer):
    prompt = ''

    for index in example_indices_full:
        raw_question = dataset['train'][index]['Question']
        answer = dataset['train'][index]['Answer']

        question = raw_question.replace("### Question:", "").replace("### Answer:", "").strip()

        prompt += f"""
Define all data science terms and concepts. Explain the concept in simple terms.


Question:
{question}

Answer:
{answer}


"""

    raw_question = dataset['train'][example_index_to_answer]['Question']
    question = raw_question.replace("### Question:", "").replace("### Answer:", "").strip()

    prompt += f"""
Define all data science terms and concepts. Explain the concept in simple terms.

Question:
{question}

Answer:
"""

    return prompt

In [ ]:
example_indices_full = [40] #means 1 solved example
example_index_to_answer = 45 #means new question to answer

one_shot_prompt = make_prompt(example_indices_full, example_index_to_answer)

print(one_shot_prompt)


Define all data science terms and concepts. Explain the concept in simple terms.


Question:
What is the difference between population and sample in statistics?

Answer:
Population refers to the entire group of individuals or instances about whom we are interested in making inferences...



Define all data science terms and concepts. Explain the concept in simple terms.

Question:
What is Data Science?

Answer:



In [ ]:
answer = dataset['train'][example_index_to_answer]['Answer']

inputs = tokenizer(one_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN ANSWER:\n{answer}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Data Science is a multidisciplinary field that involves extracting insights and knowledge from data...

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ONE SHOT:
Data science is the study of how people use data to make decisions.


###**Experimenting with more  Shot Inference**

Let's explore few shot inference by adding two more full dialogue-summary pairs to your prompt.

In [ ]:
example_indices_full = [40, 42, 11] #means 3 solved example
example_index_to_answer = 33 #means new question to answer

few_shot_prompt = make_prompt(example_indices_full, example_index_to_answer)

print(few_shot_prompt)


Define all data science terms and concepts. Explain the concept in simple terms.


Question:
What is the difference between population and sample in statistics?

Answer:
Population refers to the entire group of individuals or instances about whom we are interested in making inferences...



Define all data science terms and concepts. Explain the concept in simple terms.


Question:
What is the purpose of the Extract phase in ETL?

Answer:
The Extract phase in ETL involves extracting data from various sources...



Define all data science terms and concepts. Explain the concept in simple terms.


Question:
How can you assess the multicollinearity of features in a regression model?

Answer:
Multicollinearity occurs when two or more features in a regression model are highly correlated, making it challenging to distinguish their individual effects...



Define all data science terms and concepts. Explain the concept in simple terms.

Question:
How do you handle data transformation in the 

In [ ]:
#Now pass this prompt to perform a few shot inference:

answer = dataset['train'][example_index_to_answer]['Answer']

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=100,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN ANSWER:\n{answer}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Data transformation in ETL processes includes cleaning, restructuring, and enriching data...

---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
Data transformation is the process of converting data from one source to another.


In this test, **few-shot** prompting did not improve the answer much compared with **one-shot** prompting. The one-shot answer was already clearer because the model saw one full example of the expected question-and-answer style. Adding more examples can help sometimes, but after about 5 or 6 examples it usually stops helping much, and too many examples can exceed FLAN-T5’s 512-token input limit. For my interview assistant, this means I should use a few strong examples instead of stuffing the prompt with too many examples.

This exercise asks me to test few-shot inference by changing which examples the model sees before answering a new question. I can change the row numbers in example_indices_full to choose the solved examples, and change example_index_to_answer to choose the new question. I should also test different numbers of examples, such as one-shot, two-shot, or three-shot, while staying within the dataset’s 47 rows and the model’s input limit. The goal is to see whether different examples help the model give better answers.

###**Generative Configuration Parameters for Inference**

This section shows how to control the model’s answer by changing the settings inside `generate()`. So far, I only used `max_new_tokens=50`, which limits how long the answer can be. GenerationConfig helps organize settings like `do_sample`, `temperature`, `top_k`, and `top_p`, which can make the model’s answer more predictable or more creative. For my data science interview assistant, these settings help me test whether I want short, stable interview answers or more varied explanations.

In [ ]:
#generation_config = GenerationConfig(max_new_tokens=50)
#generation_config = GenerationConfig(max_new_tokens=10)
generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1, top_k = 1)
#generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=2.0)
#generation_config = GenerationConfig(max_new_tokens=100, do_sample=True, temperature=1.0)

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        generation_config=generation_config,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')
print(dash_line)
print(f'BASELINE HUMAN ANSWER:\n{answer}\n')

---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
Data transformation is the process of converting data from one source to another.
---------------------------------------------------------------------------------------------------
BASELINE HUMAN ANSWER:
Data transformation in ETL processes includes cleaning, restructuring, and enriching data...



This section shows that generation settings control how the model writes its answer. If `max_new_tokens=10`, the model is only allowed to write a very short answer, so the response may get cut off. If `do_sample=True` and the `temperature` is changed, the model has more freedom to create different answers instead of always choosing the safest wording. `Prompt engineering` can improve the answers a lot, but it has limits, so `fine-tuning` is the next step when you want the model to understand your specific task more deeply.
